# Process All Videos — CPU WavLM + GPU Training

WavLM runs on CPU (CUDA incompatible with T4).
Training runs on GPU.

**Runtime:** GPU enabled (for training) | **Time:** ~8-12 hours

In [ ]:
# Cell 1: Setup — install compatible PyTorch FIRST
import subprocess, sys
print('Installing PyTorch 2.2.0+cu118 (supports P100/sm_60)...')
r = subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'torch==2.2.0', 'torchaudio==2.2.0', 'numpy<2',
    '--index-url', 'https://download.pytorch.org/whl/cu118',
    '--extra-index-url', 'https://pypi.org/simple', '-q'
], capture_output=True, text=True)
print(f'pip exit: {r.returncode}')
if r.returncode != 0:
    print(f'STDERR: {r.stderr[:500]}')

# Now install other deps + yt-dlp
subprocess.run([sys.executable, '-m', 'pip', 'install', 'yt-dlp', '-q'], capture_output=True)

# RESTART RUNTIME NOTE: In Kaggle notebooks, we need to use the new torch
# Since we can't restart, let's check if it worked

import os, json, warnings, shutil
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from transformers import AutoModel

print(f'torch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

device = None
GPU_OK = False
if torch.cuda.is_available():
    try:
        # Test if group_norm works on GPU
        gn = torch.nn.GroupNorm(1, 64).cuda()
        _ = gn(torch.randn(1, 64, 100).cuda())
        device = torch.device('cuda')
        GPU_OK = True
        print(f'✅ GPU works! {torch.cuda.get_device_name(0)}')
    except Exception as e:
        print(f'⚠️ GPU error: {e}')

if not GPU_OK:
    device = torch.device('cpu')
    print(f'Using CPU: {device}')

WORK = '/kaggle/working/process255'
LABEL_DIR = '/kaggle/input/datasets/subhajitdas/standup4ai-en-uk-labels/labels'
os.makedirs(f'{WORK}/features', exist_ok=True)
os.makedirs(f'{WORK}/audio', exist_ok=True)

n_labels = len([f for f in os.listdir(LABEL_DIR) if f.endswith('.csv')])
print(f'Labels: {n_labels} CSVs')
assert n_labels > 100, 'Not enough labels!'

In [ ]:
# Cell 2: Video IDs (all 255)
# Full list of 255 videos with audio+EMNLP labels
# Process in batches of 25 per run to stay within time limits
BATCH_NUM = 1  # Change to 2, 3... for subsequent runs
BATCH_SIZE = 25

ALL_VIDEOS = [
    '-UPIA46hBZs','-vcKXr6WBNc','0AvUvJ_S2Os','0Pl51hxcK-o','0g7nezWZyfY',
    '0zpUnJSG0EQ','18H1aeoGybw','18rLwnvxOU0','1ILQmgHvtd4','1Uo27tH3JQ4',
    '1pPnJut3KLw','1tO9MWWOgHk','1u-pq9LLWlU','21gOjz-Xk7s','2SUfHIbT0HI',
    '2axWotdMFsw','2ql8QJWmNM8','3TgRGK1vrzs','3ZTClwMxpmM','3new05S61w4',
    '41piF6uPhXg','482LeT9UT7I','4ZiXvhSxnD4','53JXuJGmhoU','5bKcTy3zag4',
    '5cdoHY0ziVA','5gp79fSWHy0','66CyaeFWucM','6Ofc2A75zuw','76r8IcowEsE',
    '7E7la6BCpRc','7Gw1NjZ13fA','7VkAFkK3bwQ','7cBFWZDXlHA','7gRo0nF1yS0',
    '7kULz2NevT4','8CoHAczz9pY','8EUpV_qyEpc','8eYSNXOsyoo','8nltoWdciws',
    '90s9HfZhM0Y','9DwiBEVDdUE','9h7-OMYItDI','9yPco6WNYG0','AES4jzE513Y',
    'AEnlxaPVtK8','AI69HZWZ26c','A_EIL1ojfK4','Azl5GJuYqE0','B9jLEExvazc',
    'BT-WOZQ5JRc','BbBymwvs7co','Bl-PyS4f8as','BoMFeYyvYP8','C1TMo0YTDLA',
    'CNKnRGig1FM','CUEvqRwSi_c','CgeJOMi1mEU','CocEMvDXiu8','CwMCoAh1-a0'
]

# Get batch
start_idx = (BATCH_NUM - 1) * BATCH_SIZE
batch_vids = ALL_VIDEOS[start_idx:start_idx+BATCH_SIZE]
print(f'Batch {BATCH_NUM}: processing videos {start_idx+1}-{min(start_idx+BATCH_SIZE, len(ALL_VIDEOS))}')
print(f'Videos: {batch_vids}')

In [ ]:
# Cell 3: Load WavLM on CPU + Define Extractors
print('Loading WavLM on CPU...')
SR_WAVLM, SR_PROSODY = 16000, 22050
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(WAVLM_DEVICE)
wavlm.eval()
print('✓ WavLM loaded')

def prosody23(y, sr):
    f = []
    try:
        f0, vd, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0c = f0[~np.isnan(f0)]
        v = vd[~np.isnan(f0)]
        f.extend([np.mean(f0c) if len(f0c)>0 else 0,
                  np.std(f0c) if len(f0c)>0 else 0,
                  np.max(f0c) if len(f0c)>0 else 0,
                  np.min(f0c) if len(f0c)>0 else 0,
                  np.mean(v) if len(v)>0 else 0])
    except:
        f.extend([0]*5)
    hop = 512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    f.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms)-np.min(rms)])
    dur = len(y)/sr
    f.extend([dur, dur/(np.sum(rms>np.mean(rms))+1)])
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
        sf2 = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        f.extend([np.mean(sc), np.mean(sb), np.mean(sf2), np.mean(zcr), np.std(zcr)])
    except:
        f.extend([0]*5)
    try:
        yh, _ = librosa.effects.hpss(y)
        hnr = np.mean(np.abs(yh))/(np.mean(np.abs(y))+1e-8)
        f.extend([hnr, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y)), 0, 0])
    except:
        f.extend([0]*6)
    return np.array(f[:23], dtype=np.float32)

def word_features(y16, y22, t0, t1):
    dur = t1 - t0
    if dur < 0.005: return None
    s16, e16 = int(t0*SR_WAVLM), min(int(t1*SR_WAVLM), len(y16))
    c16 = y16[s16:e16]
    if len(c16) < int(0.01*SR_WAVLM): return None
    target_len = 5 * SR_WAVLM
    if len(c16) < target_len:
        c16 = np.pad(c16, (0, target_len-len(c16)))
    else:
        c16 = c16[:target_len]
    with torch.no_grad():
        inp = torch.tensor(c16/32768.0, dtype=torch.float32).unsqueeze(0)
        wemb = wavlm(inp).last_hidden_state.mean(dim=1).squeeze().numpy()
    s22, e22 = int(t0*SR_PROSODY), min(int(t1*SR_PROSODY), len(y22))
    chunk22 = y22[s22:e22]
    pros = prosody23(chunk22, SR_PROSODY)
    return np.concatenate([wemb, pros])

print('✅ Ready')

In [ ]:
# Cell 4: Download Audio + Extract Features
def dl_audio(vid):
    for ext in ['.wav', '.m4a']:
        p = f'{WORK}/audio/{vid}{ext}'
        if os.path.exists(p): return p
    base = f'{WORK}/audio/{vid}'
    cmd = ['yt-dlp', '-f', 'bestaudio',
           '-o', f'{base}.%(ext)s',
           f'https://www.youtube.com/watch?v={vid}',
           '--no-playlist', '--quiet', '--socket-timeout', '90',
           '--extract-audio', '--audio-format', 'wav']
    try:
        subprocess.run(cmd, capture_output=True, text=True, timeout=180)
        if os.path.exists(f'{base}.wav'): return f'{base}.wav'
        if os.path.exists(f'{base}.m4a'): return f'{base}.m4a'
    except: pass
    return None

DONE_FILE = f'{WORK}/features_done.json'
done = set()
if os.path.exists(DONE_FILE):
    with open(DONE_FILE) as f: done = set(json.load(f))

processed, errors, no_audio = 0, [], []
for vid in tqdm(batch_vids, desc='Processing'):
    if vid in done: continue
    
    ap = dl_audio(vid)
    if not ap:
        no_audio.append(vid)
        continue
    
    lp = f'{LABEL_DIR}/{vid}.csv'
    if not os.path.exists(lp):
        continue
    
    try:
        y22, _ = librosa.load(ap, sr=SR_PROSODY, mono=True)
        y16, _ = librosa.load(ap, sr=SR_WAVLM, mono=True)
        df = pd.read_csv(lp)
        
        feats, lbls = [], []
        for _, row in df.iterrows():
            try:
                ts = eval(str(row['timestamp']))
                t0, t1 = float(ts[0]), float(ts[1])
                feat = word_features(y16, y22, t0, t1)
                lbl = str(row.get('label', 'O')).strip()
                if feat is not None:
                    feats.append(feat)
                    lbls.append(1 if lbl in ('B','I','L') else 0)
            except:
                pass
        
        if feats:
            np.save(f'{WORK}/features/{vid}_features.npy', np.array(feats, dtype=np.float32))
            np.save(f'{WORK}/features/{vid}_labels.npy', np.array(lbls, dtype=np.int32))
            done.add(vid)
            processed += 1
            pos_rate = sum(lbls)/max(len(lbls),1)
            print(f'  ✓ {vid}: {len(feats)} words ({100*pos_rate:.1f}% laugh)')
        
        # Free memory
        del y16, y22
        import gc; gc.collect()
        
        # Save checkpoint every 5
        if processed % 5 == 0:
            with open(DONE_FILE, 'w') as f:
                json.dump(sorted(done), f)
    except Exception as e:
        errors.append((vid, str(e)[:200]))
        print(f'  ❌ {vid}: {e}')

with open(DONE_FILE, 'w') as f:
    json.dump(sorted(done), f)

print(f'\n=== BATCH {BATCH_NUM} COMPLETE ===')
print(f'Processed: {processed}/{len(batch_vids)}')
print(f'Total done so far: {len(done)}')
print(f'No audio: {len(no_audio)}')
if errors:
    print(f'Errors: {errors[:3]}')